# weight-decay-decoupled — worked example 1: One AdamW step on a scalar parameter

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `weight-decay-decoupled`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

AdamW's defining feature is that weight decay is applied directly to the parameter (`p *= (1 - lr * wd)`) BEFORE the Adam moment update, rather than folding `wd * p` into the gradient. This is the decoupled form: the decay shrinks the parameter proportionally to its magnitude, independently of the gradient's scale or direction.

## Worked solution

**Step 1 — apply decoupled weight decay first.**
The formula is `p <- p * (1 - lr * wd)`, which is the same as `p -= lr * wd * p`. In PyTorch, we write this as `p.mul_(1 - lr * wd)`. This happens BEFORE anything touches the gradient — that's what 'decoupled' means.

**Step 2 — update the moment estimates.**
Standard Adam moment update: `m = beta1*m + (1-beta1)*grad` and `v = beta2*v + (1-beta2)*grad^2`. These use the raw gradient, with no weight-decay term added. This is the key contrast with L2-into-gradient (coupled) weight decay.

**Step 3 — apply bias correction.**
Divide both moments by their respective bias corrections: `m_hat = m / (1 - beta1^step)` and `v_hat = v / (1 - beta2^step)`. This corrects for the fact that both moments are initialized to zero.

**Step 4 — apply the Adam parameter update.**
Update `p -= lr * m_hat / (sqrt(v_hat) + eps)`. Note that there is no weight-decay term here — it was already applied in Step 1.

In [ ]:
import torch as t

def adamw_step_scalar(p, grad, m, v, lr, beta1, beta2, eps, wd, step):
    """
    One AdamW step on tensors p, grad, m, v (in-place).
    step: 1-indexed int.
    """
    # Step 1: Decoupled weight-decay FIRST
    p.mul_(1 - lr * wd)
    # Step 2: Moment update (uses raw grad, no wd term)
    m.mul_(beta1).add_(grad, alpha=1 - beta1)
    v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
    # Step 3: Bias correction
    m_hat = m / (1 - beta1 ** step)
    v_hat = v / (1 - beta2 ** step)
    # Step 4: Adam update (no wd here)
    p.addcdiv_(m_hat, v_hat.sqrt().add_(eps), value=-lr)

# Exercise it
t.manual_seed(42)
p = t.tensor([0.5], dtype=t.float32)
grad = t.tensor([0.1], dtype=t.float32)
m = t.zeros(1)
v = t.zeros(1)

lr, beta1, beta2, eps, wd = 1e-3, 0.9, 0.999, 1e-8, 0.01
adamw_step_scalar(p, grad, m, v, lr, beta1, beta2, eps, wd, step=1)
print('p after step:', p.item())
print('m after step:', m.item())
print('v after step:', v.item())

# Verify against torch.optim.AdamW
pt_p = t.tensor([0.5], requires_grad=True, dtype=t.float32)
opt = t.optim.AdamW([pt_p], lr=lr, betas=(beta1, beta2), eps=eps, weight_decay=wd)
opt.zero_grad()
pt_p.grad = t.tensor([0.1])
opt.step()
print('torch AdamW p:', pt_p.item())
print('Match:', abs(p.item() - pt_p.item()) < 1e-6)